# RDT-1B Feature-level BlurIG
**What this does:** Runs ManiSkill simulation to get a real robot camera frame, then applies Blur Integrated Gradients in SigLIP patch-embedding space to show which parts of the image most influenced RDT's predicted gripper action.

**Before running:** Set `Runtime > Change runtime type > A100 GPU`

**Runtime:** ~5-10 min on A100 (first run downloads ~2GB of models)

In [ ]:
import os
if os.path.exists('/content/.deps_installed'):
    print('Already installed — skip to Cell 2.')
else:
    os.system('git clone https://github.com/thu-ml/RoboticsDiffusionTransformer')
    os.system('git clone https://github.com/rdgbrandon/rdt-igtesting')
    os.system('apt-get install -qq libvulkan1 libegl1-mesa-dev libgles2-mesa-dev libosmesa6-dev')
    os.system('pip install -q einops timm pyyaml mani-skill accelerate')
    os.system('pip install -q "numpy<2"')
    os.system('pip install -q "protobuf>=5.28.0"')
    open('/content/.deps_installed', 'w').close()
    print('Install done. Restarting runtime...')
    os.kill(os.getpid(), 9)

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

path = hf_hub_download(
    repo_id='robotics-diffusion-transformer/maniskill-model',
    filename='lang_embeds/text_embed_PickCube-v1.pt',
)
dest = '/content/rdt-igtesting/lang_embed.pt'
os.makedirs(os.path.dirname(dest), exist_ok=True)
shutil.copy(path, dest)
print('Language embedding ready.')

In [ ]:
import os, traceback
os.environ['RDT_REPO']       = '/content/RoboticsDiffusionTransformer'
os.environ['LANG_EMBED']     = 'lang_embed.pt'
os.environ['MANISKILL_TASK'] = 'PickCube-v1'
os.chdir('/content/rdt-igtesting')
!git pull -q

try:
    %run rdt_blurig.py
    print("\nDone — see rdt_blurig_output.png")
except Exception as _e:
    _tb, _msg = traceback.format_exc(), str(_e)
    print(f"\nFAILED: {type(_e).__name__}: {_msg}")
    fixes = []
    if "CUDA out of memory" in _msg:
        fixes.append("OOM — reduce N_BLURIG_STEPS or N_DDPM_STEPS at top of rdt_blurig.py")
    if "ModuleNotFoundError" in _msg:
        mod = _msg.split("'")[1] if "'" in _msg else "?"
        fixes.append(f"Missing '{mod}' — add to Cell 1 pip installs and re-run from Cell 1")
    if "Vulkan" in _tb or "sapien" in _tb.lower():
        fixes.append("Renderer error — make sure GPU runtime is selected")
    print(("\nFix: " + fixes[0]) if fixes else "\n" + _tb)

In [ ]:
from IPython.display import Image
Image('rdt_blurig_output.png')

In [ ]:
import gymnasium as gym
import mani_skill.envs
import torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image as PILImage

CHUNK_SIZE = 8    # re-predict every N steps
MAX_STEPS  = 80

rollout_env = gym.make(
    TASK,
    obs_mode='rgb',
    render_mode='rgb_array',
    control_mode='pd_joint_pos',
    sensor_configs={'width': 384, 'height': 384},
)
obs, _ = rollout_env.reset(seed=42)

frames, step, done = [], 0, False
actions_chunk, chunk_pos = None, CHUNK_SIZE  # force predict on first step

while not done and step < MAX_STEPS:
    cam_key = list(obs['sensor_data'].keys())[0]
    rgb = obs['sensor_data'][cam_key]['rgb']
    if hasattr(rgb, 'cpu'): rgb = rgb.cpu().numpy()
    frames.append(np.array(rgb).squeeze().copy())

    if chunk_pos >= CHUNK_SIZE:
        frame_now = PILImage.fromarray(frames[-1]).resize((384, 384))

        qpos = obs['agent']['qpos']
        if hasattr(qpos, 'cpu'): qpos = qpos.cpu().numpy()
        qpos = np.array(qpos).flatten()
        joints_8    = np.concatenate([qpos[:7], [float(qpos[7:9].mean())]])
        joints_n    = (torch.tensor(joints_8) - STATE_MIN) / (STATE_MAX - STATE_MIN).clamp(min=1e-6) * 2 - 1
        state_now   = torch.zeros(128, dtype=DTYPE, device=DEVICE)
        state_now[MANISKILL_INDICES] = joints_n.to(DTYPE).to(DEVICE)
        proprio_now = state_now.unsqueeze(0).unsqueeze(0)
        amask_now   = torch.zeros(1, 1, 128, dtype=DTYPE, device=DEVICE)
        amask_now[0, 0, MANISKILL_INDICES] = 1.0

        with torch.no_grad():
            emb_now      = encode_image(frame_now)
            img_cond_now = rdt.img_adaptor(emb_now.repeat(1, 6, 1))
            st_now       = rdt.state_adaptor(torch.cat([proprio_now, amask_now], dim=2))
            pred         = rdt.conditional_sample(
                lang_cond, lang_attn_mask, img_cond_now,
                st_now, amask_now, ctrl_freqs,
            )  # (1, 64, 128)

        acts_8        = pred[0, :, MANISKILL_INDICES].cpu().float()
        acts_denorm   = (acts_8 + 1) / 2 * (STATE_MAX - STATE_MIN) + STATE_MIN
        actions_chunk = acts_denorm.numpy()
        chunk_pos     = 0
        print(f"  step {step:3d}: re-predicted")

    act = actions_chunk[chunk_pos]
    action_8 = act.reshape(1, 8)  # (1, 8): batch dim required by ManiSkill3
    obs, _, terminated, truncated, info = rollout_env.step(action_8)
    done = terminated or truncated
    chunk_pos += 1
    step += 1

rollout_env.close()
print(f"\nDone: {step} steps  success={info.get('success', '?')}")

n_show   = min(len(frames), 12)
indices  = np.linspace(0, len(frames) - 1, n_show, dtype=int)
fig, axes = plt.subplots(2, n_show // 2, figsize=(3.5 * (n_show // 2), 7))
for idx, ax in zip(indices, axes.flat):
    ax.imshow(frames[idx])
    ax.set_title(f"step {idx}", fontsize=8)
    ax.axis("off")

fig.suptitle(f"RDT-1B rollout: {TASK_TEXT} | {step} steps executed", fontsize=11)
plt.tight_layout()
plt.savefig("rdt_rollout_frames.png", dpi=150, bbox_inches="tight")
print("Saved: rdt_rollout_frames.png")

from IPython.display import Image as IPyImage
IPyImage("rdt_rollout_frames.png")

In [ ]:
import torch, matplotlib.pyplot as plt, numpy as np
from huggingface_hub import hf_hub_download

null_tasks = {
    "push cube":    "PushCube-v1",
    "stack cubes":  "StackCube-v1",
    "insert peg":   "PegInsertionSide-v1",
}

# ── Score check (fast) ────────────────────────────────────────────────────────
# If all scores are identical, language is not reaching the gradient.
# If scores differ but heatmaps are the same, the gradient pattern is task-invariant.
print("Score sensitivity check (no BlurIG — just one forward pass each):")
_E = single_emb.detach().requires_grad_(True)
with torch.enable_grad():
    print(f"  {TASK_TEXT:22s}  score={rdt_score(_E).item():.5f}  (real task)")

_lang_cache = {}
for task_desc, task_id in null_tasks.items():
    path = hf_hub_download(
        repo_id='robotics-diffusion-transformer/maniskill-model',
        filename=f'lang_embeds/text_embed_{task_id}.pt',
    )
    ld = torch.load(path, map_location="cpu", weights_only=False)
    lt = ld if not isinstance(ld, dict) else ld["embeddings"]
    if lt.dim() == 2:
        lt = lt.unsqueeze(0)
    lt = lt.to(DEVICE, dtype=DTYPE)
    lm = torch.ones(lt.shape[:2], dtype=torch.bool, device=DEVICE)
    with torch.no_grad():
        lc = rdt.lang_adaptor(lt)
    _lang_cache[task_desc] = (lc, lm)

    _lc, _lm = lang_cond, lang_attn_mask
    lang_cond, lang_attn_mask = lc, lm
    with torch.enable_grad():
        s = rdt_score(_E)
    lang_cond, lang_attn_mask = _lc, _lm
    print(f"  {task_desc:22s}  score={s.item():.5f}")

print()

# ── Full BlurIG for each task language ────────────────────────────────────────
null_results = {}
for task_desc, (lc, lm) in _lang_cache.items():
    _lc, _lm = lang_cond, lang_attn_mask
    lang_cond, lang_attn_mask = lc, lm
    print(f"BlurIG: same frame + '{task_desc}' ...")
    null_results[task_desc] = feature_blur_ig(single_emb)
    lang_cond, lang_attn_mask = _lc, _lm

all_cases = {TASK_TEXT: attr} | null_results
img_np    = np.array(frame_pil) / 255.0
n         = len(all_cases)

fig, axes = plt.subplots(2, n, figsize=(4.5 * n, 8))
for col, (name, at) in enumerate(all_cases.items()):
    amap = to_map(at, 384, 384)
    axes[0, col].imshow(img_np)
    axes[0, col].set_title(name, fontsize=10)
    axes[1, col].imshow(img_np)
    axes[1, col].imshow(amap, cmap="inferno", alpha=0.6, vmin=0, vmax=1)
    axes[1, col].set_title("attribution", fontsize=9)

for ax in axes.flat:
    ax.axis("off")

fig.suptitle(
    f"RDT-1B BlurIG — same frame, different task language | Image: {TASK}",
    fontsize=10,
)
plt.tight_layout()
plt.savefig("rdt_blurig_null_comparison.png", dpi=150, bbox_inches="tight")
print("Saved: rdt_blurig_null_comparison.png")

from IPython.display import Image as IPyImage
IPyImage("rdt_blurig_null_comparison.png")

In [ ]:
import torch, matplotlib.pyplot as plt, numpy as np

JOINT_NAMES = [
    "base rot (j0)", "shoulder (j1)", "upper arm (j2)", "elbow (j3)",
    "forearm rot (j4)", "wrist pitch (j5)", "wrist rot (j6)", "gripper",
]

def blurig_per_joint(emb):
    sigmas = torch.linspace(SIGMA_MAX, 0.0, N_BLURIG_STEPS + 1)
    n      = len(MANISKILL_INDICES)
    totals = [torch.zeros_like(emb) for _ in range(n)]
    _midx  = torch.tensor(MANISKILL_INDICES, device=DEVICE)

    for k in range(N_BLURIG_STEPS):
        E_t    = embed_blur(emb.detach(), sigmas[k].item()).requires_grad_(True)
        E_next = embed_blur(emb.detach(), sigmas[k + 1].item())

        with torch.enable_grad():
            img_cond = rdt.img_adaptor(E_t.repeat(1, 6, 1))
            actions  = rdt.conditional_sample(
                lang_cond, lang_attn_mask, img_cond,
                state_traj, action_mask, ctrl_freqs,
            )

        joint_scores = actions[:, :SCORE_HORIZON, :][:, :, _midx].norm(dim=(0, 1))

        for j in range(n):
            g = torch.autograd.grad(joint_scores[j], E_t, retain_graph=(j < n - 1))[0]
            totals[j] = totals[j] + g.detach() * (E_next - E_t.detach())

        if k == 0 or (k + 1) % 5 == 0:
            print(f"  step {k+1:2d}/{N_BLURIG_STEPS}  "
                  f"scores: {joint_scores.detach().cpu().float().numpy().round(3)}")
    return totals

print(f"Per-joint BlurIG ({N_BLURIG_STEPS} steps × {len(MANISKILL_INDICES)} joints) ...")
joint_attrs = blurig_per_joint(single_emb)

img_np = np.array(frame_pil) / 255.0
n      = len(MANISKILL_INDICES)

fig, axes = plt.subplots(1, n + 1, figsize=(4 * (n + 1), 4.5))
axes[0].imshow(img_np)
axes[0].set_title("input frame", fontsize=9)
axes[0].axis("off")

for j, (name, at) in enumerate(zip(JOINT_NAMES, joint_attrs)):
    amap = to_map(at, 384, 384)
    axes[j + 1].imshow(img_np)
    axes[j + 1].imshow(amap, cmap="inferno", alpha=0.6, vmin=0, vmax=1)
    axes[j + 1].set_title(name, fontsize=8)
    axes[j + 1].axis("off")

fig.suptitle(f"RDT-1B — per-joint BlurIG | Task: {TASK}", fontsize=10)
plt.tight_layout()
plt.savefig("rdt_blurig_per_joint.png", dpi=150, bbox_inches="tight")
print("Saved: rdt_blurig_per_joint.png")

from IPython.display import Image as IPyImage
IPyImage("rdt_blurig_per_joint.png")

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from transformers import AutoTokenizer

N_IG_STEPS = 50

tokenizer  = AutoTokenizer.from_pretrained("t5-small")
L_emb      = lang_tokens.shape[1]

# Try the full task description first, then the short label
_candidates = [TASK_DESCRIPTION, TASK_TEXT]
token_ids   = None
for _text in _candidates:
    _ids = tokenizer(_text, return_tensors="pt").input_ids[0]
    if len(_ids) == L_emb:
        token_ids  = _ids
        used_text  = _text
        break

if token_ids is None:
    print(f"No candidate matched embedding length {L_emb}. Using positional labels.")
    plot_labels = [f"t{i}" for i in range(L_emb)]
    group_map   = None
else:
    raw_toks = tokenizer.convert_ids_to_tokens(token_ids)
    decoded  = [tokenizer.decode([tid]).strip() or f"[{tid.item()}]" for tid in token_ids]
    print(f"Text: \"{used_text}\"")
    print(f"Tokens ({L_emb}): {decoded}")

    # Group subword tokens into words using ▁ (SentencePiece word-start marker)
    groups, cur = [], [0]
    for i in range(1, len(raw_toks)):
        if raw_toks[i].startswith('▁'):   # ▁ = new word
            groups.append(cur)
            cur = [i]
        else:
            cur.append(i)
    groups.append(cur)

    plot_labels = [''.join(decoded[i] for i in g) for g in groups]
    group_map   = groups
    print(f"Words ({len(plot_labels)}): {plot_labels}")

_midx = torch.tensor(MANISKILL_INDICES, device=DEVICE)

def rdt_score_lang(lang_interp):
    with torch.enable_grad():
        lc      = rdt.lang_adaptor(lang_interp)
        ic      = rdt.img_adaptor(single_emb.detach().repeat(1, 6, 1))
        actions = rdt.conditional_sample(
            lc, lang_attn_mask, ic,
            state_traj, action_mask, ctrl_freqs,
        )
    return actions[:, :SCORE_HORIZON, :][:, :, _midx].norm()

baseline = torch.zeros_like(lang_tokens)
delta    = lang_tokens - baseline
total    = torch.zeros_like(lang_tokens)

print(f"\nToken IG ({N_IG_STEPS} steps) ...")
for k in range(N_IG_STEPS):
    alpha  = (k + 0.5) / N_IG_STEPS
    interp = (baseline + alpha * delta).requires_grad_(True)
    score  = rdt_score_lang(interp)
    grad   = torch.autograd.grad(score, interp)[0]
    total  = total + grad.detach() * delta.detach()
    if (k + 1) % 10 == 0:
        print(f"  step {k+1}/{N_IG_STEPS}  score={score.item():.4f}")

total = total / N_IG_STEPS

attr_tok = total.squeeze(0).float().cpu().abs().sum(-1).numpy()

# Aggregate subword attributions into word-level scores
if group_map is not None:
    plot_attr = np.array([attr_tok[g].sum() for g in group_map])
else:
    plot_attr = attr_tok

plot_norm = plot_attr / (plot_attr.max() + 1e-8)

# ── Visualize ─────────────────────────────────────────────────────────────────
cmap = plt.cm.inferno
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(max(8, len(plot_labels) * 1.6), 6),
    gridspec_kw={"height_ratios": [4, 1]},
)

bars = ax1.bar(plot_labels, plot_attr, color=[cmap(v) for v in plot_norm],
               edgecolor="white", linewidth=0.5)
for bar, v in zip(bars, plot_attr):
    ax1.text(bar.get_x() + bar.get_width() / 2, v + plot_attr.max() * 0.01,
             f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax1.set_ylabel("Attribution magnitude")
ax1.set_title(f"Word-level token IG: \"{TASK_TEXT}\"  →  all joint actions  |  {N_IG_STEPS} steps",
              fontsize=11)
ax1.tick_params(axis="x", labelsize=11, rotation=20)

ax2.imshow(plot_norm.reshape(1, -1), cmap="inferno", aspect="auto", vmin=0, vmax=1)
ax2.set_xticks(range(len(plot_labels)))
ax2.set_xticklabels(plot_labels, fontsize=10, rotation=20, ha="right")
ax2.set_yticks([])

fig.suptitle(f"RDT-1B — language word attribution | Task: {TASK}", fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig("rdt_token_ig.png", dpi=150, bbox_inches="tight")
print("Saved: rdt_token_ig.png")

from IPython.display import Image as IPyImage
IPyImage("rdt_token_ig.png")

---
## Optional: try other tasks
Available tasks and their pre-encoded lang embeds:
- `PickCube-v1` — pick up the cube
- `StackCube-v1` — stack one cube on another
- `PushCube-v1` — push the cube to a target
- `PegInsertionSide-v1` — insert a peg into a hole
- `PlugCharger-v1` — plug a charger into a socket

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

TASK = 'StackCube-v1'

path = hf_hub_download(
    repo_id='robotics-diffusion-transformer/maniskill-model',
    filename=f'lang_embeds/text_embed_{TASK}.pt',
)
dest = '/content/rdt-igtesting/lang_embed.pt'
os.makedirs(os.path.dirname(dest), exist_ok=True)
shutil.copy(path, dest)

os.environ['MANISKILL_TASK'] = TASK
os.chdir('/content/rdt-igtesting')
%run rdt_blurig.py

from IPython.display import Image
Image('rdt_blurig_output.png')

---
## Tuning BlurIG parameters
Edit these at the top of `rdt_blurig.py` before running Cell 3:

| Parameter | Default | Effect |
|---|---|---|
| `N_BLURIG_STEPS` | 20 | More steps = smoother heatmap, slower |
| `N_DDPM_STEPS` | 5 | More steps = more faithful to full RDT, slower |
| `SIGMA_MAX` | 2.0 | Higher = blurrier baseline, stronger contrast in heatmap |
| `SCORE_HORIZON` | 8 | How many future action steps to include in the score |